# 🌲 Modelagem: Random Forest Regressor

Este notebook ajusta um modelo de **Floresta Aleatória**, avaliando-o por meio das partições oficiais (*Dobras 1, 2 e 3*). Ele lida bem com relações não lineares presentes nas variáveis socioeconômicas do ENEM (Q001 a Q025).

> **Nota de reprodutibilidade:** O conjunto de teste é completamente isolado da busca de parâmetros e só é avaliado ao final.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Carregamento dos dados
arquivo = Path("dados/processed/enem_2023_lc_c07.parquet")
if not arquivo.exists():
    raise FileNotFoundError(f"Base não encontrada em {arquivo.resolve()}")

df = pd.read_parquet(arquivo)
print("Dimensões originais:", df.shape)

: 

### Preparação dos Conjuntos de Validação e Teste
Usaremos a divisão oficial de desenvolvimento e teste definida no arquivo Parquet.

In [ ]:
# Filtro dos conjuntos e separação de X e y
dev = df.loc[df["CONJUNTO"] == "desenvolvimento"].reset_index(drop=True)
teste = df.loc[df["CONJUNTO"] == "teste"].reset_index(drop=True)

X = dev.drop(columns=["NU_INSCRICAO", "CONJUNTO", "DOBRA", "NU_NOTA_LC"])
y = dev["NU_NOTA_LC"]

X_teste = teste.drop(columns=["NU_INSCRICAO", "CONJUNTO", "DOBRA", "NU_NOTA_LC"])
y_teste = teste["NU_NOTA_LC"]

# Configuração das 3 dobras oficiais para validação cruzada
dobras = [
    (np.flatnonzero(dev["DOBRA"].to_numpy() != k), np.flatnonzero(dev["DOBRA"].to_numpy() == k)) 
    for k in (1, 2, 3)
]

print("Desenvolvimento:", X.shape)
print("Teste reservado:", X_teste.shape)

### Pipeline de Pré-processamento e Modelo
Para o RF, não precisamos escalonar as numéricas rigidamente, mas aplicaremos o `OrdinalEncoder` para o questionário socioeconômico.

In [ ]:
# Definição das ordens do dicionário do ENEM
ordens = {
    "Q006": list("ABCDEFGHIJKLMNOPQ"), "Q001": list("ABCDEFGH"),
    "Q002": list("ABCDEFGH"), "Q024": list("ABCDE"), "Q025": list("AB"),
    "ESCOLARIDADE_MAX_RESP": list("ABCDEFG") + ["INDETERMINADA"],
    "TP_FAIXA_ETARIA": [str(i) for i in range(1, 21)]
}

ord = [c for c in ordens if c in X.columns]
cat = [c for c in X.select_dtypes(include=["object", "category"]).columns if c not in ord]
num = [c for c in X.columns if c not in ord + cat]

steps = []
if ord: steps.append(("ord", OrdinalEncoder(categories=[ordens[c] for c in ord], handle_unknown="use_encoded_value", unknown_value=-1), ord))
if cat: steps.append(("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat))
if num: steps.append(("num", SimpleImputer(strategy="median", add_indicator=True), num))

# Pipeline Completo
modelo_rf = Pipeline([
    ("preprocessamento", ColumnTransformer(steps)),
    # max_depth limitado para não estourar a memória e controlar overfitting
    ("modelo", RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1))
])

### Validação Cruzada (Cross-Validation) Oficial

In [ ]:
inicio_cv = time.time()

resultados_cv = cross_validate(
    modelo_rf, X, y, cv=dobras,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2",
    },
    n_jobs=1
)

tempo_cv = time.time() - inicio_cv
print("RESULTADOS — VALIDAÇÃO CRUZADA OFICIAL (3 DOBRAS)")
print(f"MAE médio:  {-resultados_cv['test_MAE'].mean():.2f}")
print(f"RMSE médio: {-resultados_cv['test_RMSE'].mean():.2f}")
print(f"R² médio:    {resultados_cv['test_R2'].mean():.4f}")
print(f"Tempo:       {tempo_cv / 60:.2f} minutos")

### Avaliação no Conjunto de Teste Reservado e Visualização

In [ ]:
inicio_final = time.time()
modelo_rf.fit(X, y) 
tempo_final = time.time() - inicio_final

previsoes_teste = modelo_rf.predict(X_teste)

mae_teste = mean_absolute_error(y_teste, previsoes_teste)
rmse_teste = np.sqrt(mean_squared_error(y_teste, previsoes_teste))
r2_teste = r2_score(y_teste, previsoes_teste)

print("RESULTADOS FINAIS — TESTE RESERVADO")
print(f"MAE:  {mae_teste:.2f}")
print(f"RMSE: {rmse_teste:.2f}")
print(f"R²:   {r2_teste:.4f}")
print(f"Tempo de ajuste em desenvolvimento: {tempo_final / 60:.2f} minutos")

# --- GRÁFICO DE DESEMPENHO ---
rng = np.random.default_rng(42)
indices = rng.choice(len(y_teste), size=min(10000, len(y_teste)), replace=False)
notas_reais = np.asarray(y_teste)[indices]
notas_previstas = previsoes_teste[indices]

limite_min = min(notas_reais.min(), notas_previstas.min())
limite_max = max(notas_reais.max(), notas_previstas.max())

plt.figure(figsize=(7, 5))
plt.scatter(notas_reais, notas_previstas, alpha=0.30, s=14, color="#2a6fbb", label="Random Forest")
plt.plot([limite_min, limite_max], [limite_min, limite_max], "--", color="#d1495b", label="Previsão perfeita")
plt.xlabel("Nota real de Linguagens")
plt.ylabel("Nota prevista")
plt.title("ENEM 2023 — Real vs Previsto (Teste)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()